<span style="color:lightgreen">

# Notebook extra 2.2

En este notebook se experimenta con los parámetros de transcripción con traducción de whisper en cascada.

En el ejercicio anterior (extra 2.1) hemos probado diferentes parámetros para la transcripción con whisper, ahora usaremos el mejor resultado de ese experimento para su traducción.

En lugar de usar NLLB, usaremos el modelo `google/madlad400-3b-mt` con diferentes configuraciones para la inferencia.

Covost2 pt-en

</span>

# Cascade ST Baseline experiment using Whisper along with NLLB and Europarl-ST (Spanish to English)

In this notebook, we are going to learn how to use Whisper and Meta's large pre-trained models [NLLB](https://huggingface.co/docs/transformers/model_doc/nllb) in a cascade ST approach on the Europarl-ST using the English-Spanish translation pair.

First, we import automatic transcriptions, references and translations (and their clean counterparts) generated by the ASR baseline experiment using Whisper

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())


%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

CONFIG = Configuration(
    model_name="google/madlad400-3b-mt",
    trans_code='<2en> ' # to indicate the model that the target language is English
)

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/notebooks_pt_en
/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/app


In [2]:
from datasets import load_dataset

raw_datasets = load_dataset("csv", data_files=os.path.join(CONFIG.RESULTS_PATH, "Extra2.1-L4.1_ASR_Whisper_Baseline_Covost2_pt-en.csv"))

# Filter out rows with null values in hypothesis or translation
raw_datasets = raw_datasets.filter(lambda x: x["hypothesis"] is not None and x["translation"] is not None)

print(raw_datasets)

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/4023 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'hypothesis', 'reference', 'translation', 'hypothesis_clean', 'reference_clean', 'translation_clean'],
        num_rows: 4023
    })
})


Show the first sample for each dataset feature

In [3]:
print(raw_datasets["train"][:1]["hypothesis"])
print(raw_datasets["train"][:1]["reference"])
print(raw_datasets["train"][:1]["translation"])
print(raw_datasets["train"][:1]["hypothesis_clean"])
print(raw_datasets["train"][:1]["reference_clean"])
print(raw_datasets["train"][:1]["translation_clean"])

[' e dia de ir emprestado as pessoas daudei.']
['Pedir dinheiro emprestado às pessoas da aldeia']
['Borrow money from people in the village']
[' e dia de ir emprestado as pessoas daudei ']
['pedir dinheiro emprestado às pessoas da aldeia']
['borrow money from people in the village']


<p style="page-break-after:always;"></p>

Now we load the pre-trained tokenizer for the NLLB model and apply it to the Spanish-English pair:

In [4]:
from maikol_utils.print_utils import print_color
from transformers import AutoTokenizer

max_tok_length = 275
# checkpoint = "facebook/nllb-200-distilled-600M"
checkpoint = CONFIG.model_name 
print_color(f"The model checkpoint is: {checkpoint}")

# from flores200_codes import flores_codes
# tokenizer = AutoTokenizer.from_pretrained(
#     checkpoint, 
#     padding=True, 
#     pad_to_multiple_of=8, 
#     src_lang=CONFIG.src_code, 
#     tgt_lang=CONFIG.tgt_code, 
#     truncation=False, 
#     max_length=max_tok_length,
#     )
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint,
    padding=True,
    truncation=False,
    max_length=max_tok_length,
)

The model checkpoint is: google/madlad400-3b-mt


We will use the [Dataset.map() function](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.map). This allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset.

In our case, each sample pair is going to be preprocessed according to the training needs of the model that is to be used. In this case we tokenize speech transcriptions and translations to perform ST cascade experiments

In [5]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["hypothesis"], 
        text_target = sample["translation"],
        truncation=True,
        max_length=max_tok_length
        )
    return model_inputs

Now, we can apply the preprocess_function to the raw datasets

In [6]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/4023 [00:00<?, ? examples/s]

<p style="page-break-after:always;"></p>

bitsandbytes is a quantization library with a Transformers integration. With this integration, you can quantize a model to 8 or 4-bits and enable many other options by configuring the BitsAndBytesConfig class. For example, you can:

<ul>
<li>set load_in_4bit=True to quantize the model to 4-bits when you load it</li>
<li>set bnb_4bit_quant_type="nf4" to use a special 4-bit data type for weights initialized from a normal distribution</li>
<li>set bnb_4bit_use_double_quant=True to use a nested quantization scheme to quantize the already quantized weights</li>
<li>set bnb_4bit_compute_dtype=torch.bfloat16 to use bfloat16 for faster computation</li>
</ul>

In [7]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

Pass the quantization_config to the from_pretrained method.

In [8]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    quantization_config=quantization_config,
    # device_map="auto"
)

<p style="page-break-after:always;"></p>

## Inference

At inference time, it is recommended to use the [generate function](https://huggingface.co/docs/transformers/main_classes/text_generation). This method takes care of encoding the input and auto-regressively generates the decoder output. Check out [this blog post](https://huggingface.co/blog/how-to-generate) to know all the details about generating text with Transformers.
There’s also [this blog post](https://huggingface.co/blog/encoder-decoder#encoder-decoder) which explains how generation works in general in encoder-decoder models.

Let us first load the default inference parameters of NLLB.

In [9]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
)

print(generation_config)

GenerationConfig {
  "decoder_start_token_id": 0,
  "eos_token_id": 2,
  "pad_token_id": 1
}



In [10]:
test_batch_size = 32
# Note: 'train' here refers to the split name from CSV loading, which is actually test data
batch_tokenized_test = tokenized_datasets['train'].batch(test_batch_size)


Batching examples:   0%|          | 0/4023 [00:00<?, ? examples/s]

<span style="color:lightgreen">
Definición experimentos
</span>

In [11]:
experiments = [
    {"name": "Default (Greedy)", "params": {}}, # Uses defaults from generation_config
    {"name": "Beam Search (n=5)", "params": {"num_beams": 5, "early_stopping": True}},
    {"name": "Multinomial Sampling (T=0.7)", "params": {"do_sample": True, "temperature": 0.7}},
    {"name": "Nucleus Sampling (p=0.9)", "params": {"do_sample": True, "top_p": 0.9, "temperature": 1.0}},
    # Contrastive Search removed - not compatible with T5-based models like MADLAD-400
]


<p style="page-break-after:always;"></p>

Processing in batches to add padding and convert to tensors, then perform inference with num_beams = 1 and do_sample = False, that is, greedy search.

<span style="color:lightgreen">

### EL ÚLTIMO EXPERIMENTO NO ERA COMPATIBLE CON EL MODELO -> Output da error pero el resto están bien ejecutados
<span>

In [12]:
from tqdm import tqdm
from maikol_utils.print_utils import print_color, print_separator
from whisper.normalizers.basic import BasicTextNormalizer
from evaluate import load


all_results = {}
metric = load("sacrebleu")
comet_metric = load('comet')
normalizer = BasicTextNormalizer()
number_of_batches = len(batch_tokenized_test["hypothesis"])

references = tokenizer.batch_decode(tokenized_datasets["train"]["labels"], skip_special_tokens=True)
references_clean = [normalizer(text) for text in references]

for exp in experiments:
    print_separator(exp['name'], sep_type="SUPER")
    output_sequences = []
    
    for i in tqdm(range(number_of_batches)):
        inputs = tokenizer(
            [CONFIG.trans_code + text for text in batch_tokenized_test["hypothesis"][i]], 
            max_length=max_tok_length, 
            truncation=False, 
            return_tensors="pt", 
            padding=True,
        )
        
        # We merge generation_config with experiment params
        # Note: forced_bos_token_id is removed as it's not needed for MADLAD-400
        output_batch = model.generate(
            input_ids=inputs["input_ids"].cuda(), 
            attention_mask=inputs["attention_mask"].cuda(), 
            generation_config=generation_config,
            max_length=max_tok_length, 
            **exp["params"]
        )
        output_sequences.extend(output_batch.cpu())
    

    decoded_preds = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)
    decoded_preds_clean = [normalizer(text) for text in decoded_preds]

    bleu_score = metric.compute(predictions=decoded_preds, references=references)
    bleu_score_clean = metric.compute(predictions=decoded_preds_clean, references=references_clean)
    print_color(f'BLEU score:       {bleu_score["score"]:.2f}', color="cyan")
    print_color(f'BLEU score Clean: {bleu_score_clean["score"]:.2f}', color="cyan")

    comet_score = comet_metric.compute(predictions=decoded_preds, references=references, sources=raw_datasets["train"]["hypothesis"])
    comet_score_clean = comet_metric.compute(predictions=decoded_preds_clean, references=references_clean, sources=raw_datasets["train"]["hypothesis"])
    print_color(f"COMET score:       {comet_score['mean_score']:.2%}", color="cyan")
    print_color(f"COMET score Clean: {comet_score_clean['mean_score']:.2%}", color="cyan")


    all_results[exp["name"]] = {}
    all_results[exp["name"]]["output"] = output_sequences
    all_results[exp["name"]]["BLEU"] = bleu_score["score"]
    all_results[exp["name"]]["BLEU_CLEAN"] = bleu_score_clean["score"]
    all_results[exp["name"]]["COMET"] = comet_score['mean_score']
    all_results[exp["name"]]["COMET_CLEAN"] = comet_score_clean['mean_score']


/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


Encoder model frozen.


/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/venv/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


...

TPU available: False, using: 0 TPU cores


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


COMET score:       74.29%
COMET score Clean: 75.68%


In [13]:
# Find best scores
best_bleu = max(all_results.items(), key=lambda x: x[1]["BLEU"])
best_bleu_clean = max(all_results.items(), key=lambda x: x[1]["BLEU_CLEAN"])
best_comet = max(all_results.items(), key=lambda x: x[1]["COMET"])
best_comet_clean = max(all_results.items(), key=lambda x: x[1]["COMET_CLEAN"])

print_separator("BEST SCORES", sep_type="SUPER")

print_color(f"Best BLEU Score (Non-normalized):", color="yellow")
print_color(f" - Experiment: {best_bleu[0]}", color="cyan")
print_color(f" - Score: {best_bleu[1]['BLEU']:.2f}", color="green")

print_color(f"Best BLEU Score (Normalized):", color="yellow")
print_color(f" - Experiment: {best_bleu_clean[0]}", color="cyan")
print_color(f" - Score: {best_bleu_clean[1]['BLEU_CLEAN']:.2f}", color="green")

print_color(f"Best COMET Score (Non-normalized):", color="yellow")
print_color(f" - Experiment: {best_comet[0]}", color="cyan")
print_color(f" - Score: {best_comet[1]['COMET']:.2%}", color="green")

print_color(f"Best COMET Score (Normalized):", color="yellow")
print_color(f" - Experiment: {best_comet_clean[0]}", color="cyan")
print_color(f" - Score: {best_comet_clean[1]['COMET_CLEAN']:.2%}", color="green")

# Show two translation examples from the best overall experiment (using best BLEU clean as criterion)
print_separator("TRANSLATION EXAMPLES", sep_type="SUPER")
print_color(f"\nShowing examples from: {best_bleu_clean[0]}", color="yellow")

best_outputs = best_bleu_clean[1]["output"]
best_translations = tokenizer.batch_decode(best_outputs[:2], skip_special_tokens=True)

for idx, (source, reference, translation) in enumerate(zip(
    raw_datasets["train"]["hypothesis"][:2],
    references[:2],
    best_translations
), 1):
    print_color(f"\nExample {idx}:", color="magenta")
    print_color(f"  Source (PT):      {source}", color="white")
    print_color(f"  Reference (EN):   {reference}", color="white")
    print_color(f"  Translation (EN): {translation}", color="cyan")

                                                          BEST SCORES                                                           

Best BLEU Score (Non-normalized):
 - Experiment: Beam Search (n=5)
 - Score: 39.23
Best BLEU Score (Normalized):
 - Experiment: Beam Search (n=5)
 - Score: 41.27
Best COMET Score (Non-normalized):
 - Experiment: Beam Search (n=5)
 - Score: 75.52%
Best COMET Score (Normalized):
 - Experiment: Beam Search (n=5)
 - Score: 77.04%
                                                      TRANSLATION EXAMPLES                                                      


Showing examples from: Beam Search (n=5)

Example 1:
  Source (PT):       e dia de ir emprestado as pessoas daudei.
  Reference (EN):   Borrow money from people in the village
  Translation (EN): and day to go borrowed people were.

Example 2:
  Source (PT):       Tronca-vos.
  Reference (EN):   Lock them up
  Translation (EN): Cut it off.
